你提到的 **“AMAR”** 应该是个笔误，根据你提供的PDF目录图片以及数学建模的惯例，你要找的应该是 **ARMA模型 (自回归滑动平均模型)**。

在实际应用中，单纯的 ARMA 只能处理平稳数据。如果数据有趋势（一直在涨），我们需要对数据进行差分处理，这就变成了 **ARIMA 模型 (差分自回归滑动平均模型)**。

**ARIMA 是 ARMA 的完全体**。为了让你能直接解决问题，下面的代码和讲解我将以 **ARIMA** 为主（因为把它里面的差分参数 $d$ 设为 0，它就是 ARMA）。

---

### 一、 算法原理
**核心思想**：**“今天的数值，取决于‘昨天的数值’和‘昨天的意外’。”**

ARMA/ARIMA 将时间序列分解为三个部分：
1.  **AR (AutoRegressive, 自回归)**：**“历史的惯性”**。今天的股价跟昨天的股价有关。如果 $p=2$，说明今天跟昨天、前天都有关。
2.  **I (Integrated, 差分)**：**“去趋势”**。如果数据一直在涨（不平稳），我就减去昨天的值，看“增量”是不是平稳的。这是为了满足模型对**平稳性**的要求。
3.  **MA (Moving Average, 滑动平均)**：**“历史的干扰”**。今天的数值不仅受过去数值影响，还受过去预测误差（突发消息、噪音）的影响。如果 $q=1$，说明昨天的“意外”还在影响今天。

---

### 二、 推导与步骤 (ARIMA)

模型记作 $ARIMA(p, d, q)$。

#### 1. 数据平稳性检验 (ADF Test)
时间序列模型有一个硬性要求：**数据必须是平稳的**（均值和方差不随时间变化）。
*   如果不平稳，就进行 $d$ 阶差分（通常 $d=1$ 就够了，就是每一天减去前一天）。

#### 2. 定阶 (确定 p 和 q)
这是最难的一步。
*   **方法一（看图）**：看 ACF（自相关图）和 PACF（偏自相关图）的截尾性质。
    *   PACF 突然截断 -> 确定 $p$。
    *   ACF 突然截断 -> 确定 $q$。
*   **方法二（智能枚举）**：计算不同 $(p, q)$ 组合下的 **AIC / BIC 准则**（信息量准则）。AIC 值越小，说明模型越好（既准又简单）。**代码中我将使用这个方法。**

#### 3. 模型公式
$$ y_t = \underbrace{c + \phi_1 y_{t-1} + \dots}_{\text{AR部分}} + \underbrace{\theta_1 \epsilon_{t-1} + \dots}_{\text{MA部分}} + \epsilon_t $$

#### 4. 白噪声检验
模型建好后，剩下的残差（预测值 - 真实值）应该是一堆毫无规律的**白噪声**。如果残差还有规律，说明模型没提取干净。

---

### 三、 适用场景
1.  **短期预测**：非常适合股票、汇率、气温等短期数据的预测。
2.  **数据量适中**：至少要有 30-50 个数据点，太少没法抓规律。
3.  **平稳或有趋势**：如果是S型增长用Logistic，如果是指数增长用灰色预测，如果是**波动式上升/下降**，用 ARIMA。
4.  **无明显季节性**：如果有明显的周期（比如每年6月都高），需要用 **SARIMA**（季节性ARIMA）。

---

### 四、 Python 代码框架

这里我使用 `statsmodels` 库。为了让你省去“看图定阶”的痛苦，我写了一个**自动寻找最优 (p, d, q) 参数**的逻辑（基于 AIC 准则）。

```python
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import warnings

# 忽略数值计算中的警告
warnings.filterwarnings("ignore")
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

class ArimaPredictor:
    def __init__(self, data):
        """
        :param data: 一维时间序列数据 (list 或 numpy array)
        """
        self.data = pd.Series(data)
        self.best_order = None # (p, d, q)
        self.model = None

    def check_stationarity(self):
        """
        ADF 检验数据平稳性
        :return: 建议的差分阶数 d
        """
        # 0阶差分检验
        adf_res = adfuller(self.data)
        p_value = adf_res[1]
        print(f"原始数据 ADF p-value: {p_value:.4f}")

        if p_value < 0.05:
            print(">> 数据平稳，建议 d=0")
            return 0

        # 1阶差分检验
        diff1 = self.data.diff().dropna()
        adf_res1 = adfuller(diff1)
        p_value1 = adf_res1[1]
        print(f"一阶差分 ADF p-value: {p_value1:.4f}")

        if p_value1 < 0.05:
            print(">> 一阶差分后平稳，建议 d=1")
            return 1
        else:
            print(">> 一阶差分仍不平稳，建议 d=2")
            return 2

    def auto_fit(self, max_p=3, max_q=3):
        """
        网格搜索寻找最优 (p, d, q) 组合，依据 AIC 准则
        """
        # 1. 确定 d
        d = self.check_stationarity()

        best_aic = float('inf')
        best_cfg = None

        print("-" * 30)
        print(f"开始寻找最优 p, q (d={d})...")

        # 2. 遍历 p 和 q
        # 注意：p和q一般不会很大，0到3通常足够
        for p in range(max_p + 1):
            for q in range(max_q + 1):
                if p == 0 and q == 0:
                    continue
                try:
                    model = ARIMA(self.data, order=(p, d, q))
                    res = model.fit()
                    if res.aic < best_aic:
                        best_aic = res.aic
                        best_cfg = (p, d, q)
                        self.model = res
                except:
                    continue

        self.best_order = best_cfg
        print(f"最优参数找到: (p, d, q) = {self.best_order}, AIC = {best_aic:.2f}")
        print("-" * 30)

    def predict(self, steps=5):
        """
        预测未来
        """
        if self.model is None:
            raise ValueError("模型未训练，请先调用 auto_fit()")

        # forecast 返回的是未来 steps 步的预测
        forecast_res = self.model.get_forecast(steps=steps)
        pred_values = forecast_res.predicted_mean

        # 获取置信区间 (默认95%)
        conf_int = forecast_res.conf_int(alpha=0.05)

        return pred_values, conf_int

    def plot_results(self, steps=5):
        """
        绘图展示
        """
        preds, conf = self.predict(steps)

        plt.figure(figsize=(10, 6))

        # 画历史数据
        plt.plot(self.data.index, self.data, label='历史数据')

        # 画预测数据
        future_index = range(len(self.data), len(self.data) + steps)
        plt.plot(future_index, preds, color='red', marker='o', label='未来预测')

        # 画置信区间 (阴影部分)
        plt.fill_between(future_index,
                         conf.iloc[:, 0],
                         conf.iloc[:, 1],
                         color='pink', alpha=0.3, label='95%置信区间')

        plt.title(f'ARIMA{self.best_order} 时间序列预测')
        plt.legend()
        plt.grid(True)
        plt.show()

# ================= 使用示例 =================

if __name__ == "__main__":
    # 场景：模拟某股票收盘价 (带一点趋势和随机波动)
    np.random.seed(42)
    x = np.linspace(0, 10, 50)
    # 构造数据: 线性趋势 + 周期 + 随机噪声
    y = 2 * x + 3 * np.sin(x) + np.random.normal(0, 1, 50)

    # 1. 初始化模型
    arima = ArimaPredictor(y)

    # 2. 自动训练 (自动定阶)
    arima.auto_fit(max_p=4, max_q=4)

    # 3. 预测未来 5 个时间点
    future_preds, conf_interval = arima.predict(steps=5)

    print("未来5期预测值:")
    print(future_preds)

    # 4. 绘图
    arima.plot_results(steps=10)
```

### 代码使用的“修修补补”指南：

1.  **关于 ARMA 与 ARIMA**：
    *   如果你的数据**非常平稳**（ADF检验 $p < 0.05$），代码会自动把 $d$ 设为 0。这时候跑的就是纯正的 **ARMA** 模型。
    *   如果数据在涨，它会自动设 $d=1$，跑的就是 **ARIMA**。
    *   *怎么选？* 直接用这个代码，它自适应，不用你操心。

2.  **关于定阶 (p, q)**：
    *   代码里用了简单的网格搜索（Grid Search），遍历 0 到 4 的组合。
    *   *运行慢怎么办？* 如果数据量很大（几千行），`auto_fit` 会很慢。此时可以将 `max_p` 和 `max_q` 改小一点（比如2）。
    *   *AIC准则可靠吗？* 绝大多数情况可靠。如果想更严谨，可以结合 BIC 准则。

3.  **预测结果是一条直线？**
    *   如果你发现 ARIMA 预测出来的未来曲线是一条**直线**（或者很快收敛成直线），不要慌，这是正常的。
    *   ARIMA 擅长捕捉**短期的惯性**。随着预测步数增加，历史信息的影响力衰减，预测值就会回归到均值（或者沿着趋势线走）。
    *   *结论*：ARIMA **不适合做长期预测**（比如只用30个点去预测未来50个点），只适合预测接下来 3-5 个点。

4.  **遇到季节性数据怎么办？**
    *   如果你看数据图，发现有明显的波浪（比如每年12月销量猛增）。
    *   普通的 ARIMA 搞不定这个，需要用 **SARIMA**。
    *   *怎么修？* 在 `statsmodels` 中，使用 `SARIMAX` 函数，它多了一个 `seasonal_order=(P,D,Q,s)` 参数。这属于进阶内容，通常比赛中如果没强调季节性，ARIMA 足够应付。